In [1]:
import pandas as pd

In [3]:
df = pd.read_csv('vehicle_service_reviews_sentiments.csv')
df.head(5)

,text,sentiments
0,Great staff always helps and always nice. Alwa...,Good
1,Took my vehicle here for some work a few years...,Good
2,"First time here and they did a great job, very...",Good
3,A GREAT EXPERIENCE!!!!!!!!! I was a completel...,Good
4,OMG!!! I can't believe this kind of service ex...,Bad


In [4]:
import nltk
import string
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords

In [5]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
ps = PorterStemmer()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [6]:
import re
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))
negation_words = {
    'not', 'no', 'never', 'neither', 'nor', 'but', 'cannot',
    'barely', 'hardly', 'scarcely', 'without', 'against'
}
contraction_remnants = {
    'don', 't', 'doesn', 'didn', 'wasn', 'weren', 'haven', 'hasn',
    'hadn', 'won', 'wouldn', 'shann', 'shouldn', 'can', 'couldn',
    'isn', 'aren', 'ain'
}
all_negations_to_keep = negation_words.union(contraction_remnants)
stop_words = stop_words - all_negations_to_keep

def clean_text(text):
    # Handle missing/null values safely
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # URL removal
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Remove punctuation and special characters BEFORE tokenization
    text = re.sub(f'[{re.escape(string.punctuation)}]', ' ', text)

    # Remove numbers
    # Prevents mileages, prices, and years from clogging up vocabulary
    text = re.sub(r'\d+', '', text)

    # Tokenize
    words = word_tokenize(text)

    # Filter out stopwords
    filtered_and_stemmed_words = []
    for word in words:
        # Strip extra whitespace just in case
        word = word.strip()
        # Filter out stopwords and single-letter junk tokens
        if word and (word not in stop_words) and len(word) > 1:
            filtered_and_stemmed_words.append(ps.stem(word))

    return " ".join(filtered_and_stemmed_words)

In [7]:
df["clean_text"] = df["text"].apply(clean_text)
df.head(5)

,text,sentiments,clean_text
0,Great staff always helps and always nice. Alwa...,Good,great staff alway help alway nice alway clean ...
1,Took my vehicle here for some work a few years...,Good,took vehicl work year ago manufactur recal pro...
2,"First time here and they did a great job, very...",Good,first time great job satisfi car wash servic r...
3,A GREAT EXPERIENCE!!!!!!!!! I was a completel...,Good,great experi complet new custom not appoint pl...
4,OMG!!! I can't believe this kind of service ex...,Bad,omg can believ kind servic exist husband call ...


# Feature extraction

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1,2))
X = tfidf.fit_transform(df['clean_text']).toarray()

In [9]:
Y = df['sentiments'].values

# Model training

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

In [11]:
print("Shape of X (TF-IDF features):", X.shape)
print("Shape of Y (target variable):", Y.shape)

Shape of X (TF-IDF features): (229961, 3000)
Shape of Y (target variable): (229961,)


# Train data

In [12]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB()
model.fit(X_train, Y_train)

MultinomialNB()

In [15]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)
print(accuracy_score(Y_test, y_pred))
print("\nClassification Report:\n", classification_report(Y_test, y_pred))

0.9396864740286566

Classification Report:
               precision    recall  f1-score   support

         Bad       0.93      0.92      0.92     18106
        Good       0.95      0.95      0.95     27887

    accuracy                           0.94     45993
   macro avg       0.94      0.94      0.94     45993
weighted avg       0.94      0.94      0.94     45993



In [16]:
import joblib

# Define the file names
model_filename = 'multinomial_nb_model.pkl'
vectorizer_filename = 'tfidf_vectorizer.pkl'

# Save the trained model to disk
joblib.dump(model, model_filename)
print(f"Model successfully saved to {model_filename}")

# Save the vectorizer to disk
joblib.dump(tfidf, vectorizer_filename)
print(f"Vectorizer successfully saved to {vectorizer_filename}")

Model successfully saved to multinomial_nb_model.pkl
Vectorizer successfully saved to tfidf_vectorizer.pkl
